# EcoRewind 2.0 — Phase 5 analysis + demo export (Colab **CPU**, High-RAM)

Run after the GPU sections you want included (sections whose runs don't exist yet are skipped automatically).

### 1. Setup

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
TOKEN = userdata.get('GITHUB_TOKEN')
REPO = '/content/ecorewind2'
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://{TOKEN}@github.com/DAISHINKAN7/EcoRewind2.git {REPO}
%cd {REPO}
!git log --oneline -1
!pip -q install -r requirements.txt
D = '/content/drive/MyDrive/EcoRewind2'
X = '/content/drive/MyDrive/EcoRewind2_exports'
!python scripts/cubes_cache.py --drive {D}/cubes --local /content/cubes --exports-drive {X} 2>&1 | tail -4
!mkdir -p /content/ppost && cp -n {X}/*PPOST* /content/ppost/ 2>/dev/null; ls /content/ppost | wc -l
RUN = lambda run_id, site, track, models, extra='': get_ipython().system(
    f"python -u scripts/run_baselines.py --cubes /content/cubes --phase1 {D}/phase1 --out {D}/runs --run-id {run_id} "
    f"--site {site} --track {track} --models {models} --table-cache /content/tables --ppost-exports /content/ppost {extra} "
    f"2>&1 | tee -a /content/{run_id}.log")

### 2. Baselines with saved predictions (for the demo maps; ~2–3 h CPU, once)

In [ ]:
for site in ('everglades', 'mississippi'):
    for track in ('A', 'B'):
        RUN('phase2_baselines_v3_pred', site, track,
            'persistence climatology harmonic harmonic_notrend synth_control lgbm', '--save-predictions')

### 3. All analyses

In [ ]:
import glob
R = f'{D}/runs'
DEEP = ' '.join(p for p in [f'{R}/phase3_core_v1'] + sorted(glob.glob(f'{R}/phase5_*')) if os.path.isdir(p))
MAIN = f'{R}/phase2_baselines_v3 {R}/phase3_core_v1'
# Phase 4 verdict incl. the exploratory neighbour model (reported in its own rows)
NBR = f'{R}/phase5_neighbour' if os.path.isdir(f'{R}/phase5_neighbour') else ''
!python scripts/placebo_verdict.py --runs {MAIN} {NBR} --phase1 {D}/phase1 --out {D}/reports
!python scripts/attribution_spread.py --runs {MAIN} --verdict {D}/reports --out {D}/reports
!python scripts/external_evidence.py --runs {MAIN} --cubes /content/cubes --phase1 {D}/phase1 --out {D}/reports
!python scripts/carbon_illustrative.py --runs {MAIN} --verdict {D}/reports --table-cache /content/tables --out {D}/reports
!python scripts/phase5_tables.py --runs {DEEP} --out {D}/reports

### 4. Demo export (layers + JSON → Drive `EcoRewind2/demo`; rasters take ~30–60 min)

In [ ]:
!python scripts/export_demo.py --cubes /content/cubes --phase1 {D}/phase1 --table-cache /content/tables \
    --deep-runs {DEEP} --baseline-runs {R}/phase2_baselines_v3_pred --reports {D}/reports --out {D}/demo
!du -sh {D}/demo; ls {D}/demo

In [ ]:
!ls {D}/runs/phase2_baselines_v3_pred/predictions | wc -l
!ls {D}/runs/phase2_baselines_v3_pred/predictions | head

Send me `reports/external_evidence.md`, `reports/phase5_tables.md`, `reports/carbon_illustrative.md` and the new `reports/placebo_verdict.md`.

In [ ]:
import glob
for site in ('everglades','mississippi'):
    for tr in ('A','B'):
        f = glob.glob(f'{D}/runs/phase2_baselines_v3_pred/{site}*{tr}*/predictions/*.npz') \
          or glob.glob(f'{D}/runs/phase2_baselines_v3_pred/**/{site}*/**/{tr}/**/predictions/*.npz', recursive=True)
        print(site, tr, len(f))
!ls -R {D}/runs/phase2_baselines_v3_pred | head -40